# 🎓 VIGIL AI — Classroom Cheating Detection Training Pipeline
## 🚀 Multi-Platform: Google Colab (Free T4) / Kaggle (GPU P100) / Local GPU

**Project:** VIGIL AI — Automated Exam Proctoring System  
**Target:** Fine-tune YOLOv8/YOLO11 on Classroom Multi-Student Behavior Dataset (5 Classes)  
**Classes:**  
`0: back peeking` | `1: front peeking` | `2: no cheating` | `3: phone using` | `4: side peeking`  

---

### 1. ⚙️ Environment Setup & GPU Verification

In [ ]:
# Install Ultralytics and dependencies
!pip install -q ultralytics opencv-python-headless matplotlib pyyaml tqdm

import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: Running on CPU. Training will be slower. Please switch to GPU runtime if on Colab/Kaggle.")

### 2. 📂 Dataset Configuration
Specify your dataset path or download directly from Roboflow Universe.

In [ ]:
import os
from pathlib import Path
import yaml

# Create data.yaml
data_config = {
    'path': os.path.abspath('dataset'), # Root directory of dataset
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 5,
    'names': [
        'back peeking',
        'front peeking',
        'no cheating',
        'phone using',
        'side peeking'
    ]
}

os.makedirs('configs', exist_ok=True)
yaml_path = 'configs/classroom_data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"Generated dataset config at: {yaml_path}")
print(open(yaml_path).read())

### 3. 🔍 Dataset Integrity & Class Distribution Check

In [ ]:
from collections import Counter

def inspect_split(split_name):
    lbl_dir = Path(f"dataset/{split_name}/labels")
    if not lbl_dir.exists():
        print(f"Split {split_name} labels directory not found: {lbl_dir}")
        return Counter()
    
    counts = Counter()
    for txt_file in lbl_dir.glob("*.txt"):
        content = txt_file.read_text().strip()
        if not content:
            continue
        for line in content.splitlines():
            parts = line.strip().split()
            if parts:
                counts[int(parts[0])] += 1
    return counts

print("=== DATASET CLASS DISTRIBUTION ===")
for split in ['train', 'valid', 'test']:
    c = inspect_split(split)
    print(f"Split '{split:>5}': {dict(sorted(c.items()))}")

### 4. 🏋️ Model Training (YOLOv8 Nano / Small Baseline)

In [ ]:
from ultralytics import YOLO

# Initialize pretrained base model
model = YOLO('yolov8n.pt')

# Start training
results = model.train(
    data=yaml_path,
    epochs=100,
    patience=15,
    batch=16,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    # Proportional augmentations tailored for classroom context
    hsv_h=0.015,
    hsv_s=0.3,
    hsv_v=0.3,
    degrees=4.0,
    translate=0.1,
    scale=0.25,
    flipud=0.0, # Do not flip vertical
    fliplr=0.5,
    mosaic=0.8,
    mixup=0.1,
    project='runs/train',
    name='classroom_yolov8n',
    save=True,
    plots=True,
    val=True
)

print("Training completed!")

### 5. 📊 Comprehensive Per-Class Evaluation on Test Split

In [ ]:
# Load best checkpoint
best_model_path = 'runs/train/classroom_yolov8n/weights/best.pt'
eval_model = YOLO(best_model_path)

# Validate against unseen test set
metrics = eval_model.val(data=yaml_path, split='test', plots=True)

class_names = ['back peeking', 'front peeking', 'no cheating', 'phone using', 'side peeking']
print("\n" + "=" * 65)
print("         PER-CLASS TEST SET BENCHMARK REPORT")
print("=" * 65)
print(f"{'Class':<16} | {'Precision':<10} | {'Recall':<8} | {'mAP50':<8}")
print("-" * 65)
for i, cls_name in enumerate(class_names):
    p = metrics.box.p[i] if i < len(metrics.box.p) else 0.0
    r = metrics.box.r[i] if i < len(metrics.box.r) else 0.0
    map50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0.0
    print(f"{cls_name:<16} | {p:<10.4f} | {r:<8.4f} | {map50:<8.4f}")
print("=" * 65)

### 6. 🖼️ Plot Confusion Matrix & Training Metrics

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

cm_path = 'runs/train/classroom_yolov8n/confusion_matrix_normalized.png'
if os.path.exists(cm_path):
    plt.figure(figsize=(10, 8))
    plt.imshow(Image.open(cm_path))
    plt.axis('off')
    plt.title('Normalized Confusion Matrix', fontsize=14)
    plt.show()

results_path = 'runs/train/classroom_yolov8n/results.png'
if os.path.exists(results_path):
    plt.figure(figsize=(14, 10))
    plt.imshow(Image.open(results_path))
    plt.axis('off')
    plt.title('Training Loss & Metric Curves', fontsize=14)
    plt.show()

### 7. 💾 Export Best Production Weights for VIGIL AI Server

In [ ]:
import shutil

os.makedirs('models', exist_ok=True)
target_path = 'models/classroom_best.pt'
shutil.copy2(best_model_path, target_path)
print(f"✅ Exported model weights ready for VIGIL AI Server: {target_path}")

# Optional: Export to ONNX format for cross-platform high performance
# eval_model.export(format='onnx', dynamic=True)
# print("✅ ONNX model exported!")